# recount tutorial

Walk through the canonical 4-leaf binary tree example — same one used in the test suite — to demonstrate the API end-to-end.

Tree: `((A:0.5,B:0.3):0.2,(C:0.4,D:0.6):0.1);`

Node indices (leaves first, parent index > child index):
```
    0=A  1=B  2=C  3=D       (leaves)
    4=(A,B)  5=(C,D)         (internal)
    6=root
```

## 1. Set up the tree, rates, and profile data

In [ ]:
import numpy as np

from recount import Tree, GLDRates, compute_survival_params

# Parent array: -1 marks the root.
tree = Tree(parent=np.array([4, 4, 5, 5, 6, 6, -1]), leaf_names=["A", "B", "C", "D"])

rates = GLDRates(
    tree=tree,
    gain  =np.array([0.10, 0.20, 0.15, 0.25, 0.30, 0.18, 0.50]),
    loss  =np.array([1.00] * 7),
    dup   =np.array([0.40, 0.30, 0.50, 0.20, 0.45, 0.35, 0.00]),  # dup=0 at root → Poisson gain there
    length=np.array([0.50, 0.30, 0.40, 0.60, 0.20, 0.10, np.inf]),  # root edge length is +inf
)

# Family-by-leaf profile table — counts of gene copies at each leaf.
profiles = np.array([
    [1, 0, 1, 1],
    [2, 0, 0, 1],
    [1, 1, 1, 1],
    [3, 2, 1, 0],
], dtype=np.int64)

print(f"tree: {tree.num_leaves} leaves, {tree.num_nodes} nodes, root at {tree.root}")
print(f"profiles: {profiles.shape[0]} families × {profiles.shape[1]} leaves")

## 2. Inspect the derived survival parameters

The raw edge rates (μ, λ, γ, t) determine the per-edge probability parameters (p, q, r/κ). The bottom-up pass then computes:

- ε[v] = probability a single lineage at v has zero leaf descendants in v's subtree
- p̃, q̃, r̃/κ̃ — "survival" versions of (p, q, r/κ) that absorb the subtree extinction

These are the parameters the inside/outside recursion actually uses.

In [ ]:
sp = compute_survival_params(tree, rates)
print(f"{'node':>4s}  {'p_raw':>8s}  {'q_raw':>8s}  {'p~':>8s}  {'q~':>8s}  {'r̃/κ̃':>8s}  {'ε':>8s}  type")
for v in range(tree.num_nodes):
    typ = 'Pólya' if sp.is_polya[v] else 'Poisson'
    print(f"{v:>4d}  {sp.p_raw[v]:>8.4f}  {sp.q_raw[v]:>8.4f}  {sp.p[v]:>8.4f}  {sp.q[v]:>8.4f}  {sp.gain[v]:>8.4f}  {sp.eps[v]:>8.4f}  {typ}")

## 3. Compute the log-likelihood

In [ ]:
from recount import (
    log_likelihood,
    corrected_log_likelihood,
    empty_log_likelihood,
    singleton_log_likelihood,
)

print(f"raw LL                       = {log_likelihood(tree, rates, profiles):>16.10f}")
print(f"corrected LL (min_copies=1)  = {corrected_log_likelihood(tree, rates, profiles, min_copies=1):>16.10f}")
print(f"corrected LL (min_copies=2)  = {corrected_log_likelihood(tree, rates, profiles, min_copies=2):>16.10f}")
print(f"P(empty profile) in log      = {empty_log_likelihood(tree, rates):>16.10f}")
print(f"P(any singleton) in log      = {singleton_log_likelihood(tree, rates):>16.10f}")

These match the original Java implementation to ~1e-13 — see [tests/conftest.py](../tests/conftest.py) for the reference values.

## 4. Compute the analytical gradient

The gradient is in the **survival** parameterization (p̃, q̃, r̃/κ̃), as a flat array indexed `3*v + {GAIN=0, LOSS=1, DUP=2}`.

In [ ]:
from recount import gradient_survival, GAIN, LOSS, DUP

g = gradient_survival(tree, rates, profiles, min_copies=1)
print(f"{'node':>4s}  {'∂/∂GAIN':>12s}  {'∂/∂LOSS':>12s}  {'∂/∂DUP':>12s}")
for v in range(tree.num_nodes):
    print(f"{v:>4d}  {g[3*v+GAIN]:>12.6f}  {g[3*v+LOSS]:>12.6f}  {g[3*v+DUP]:>12.6f}")

Note the root (node 6) has LOSS and DUP gradients of 0 — this is because the root has `length=∞`, which forces `p̃_root = 1` (a model boundary). The gain gradient at the root is non-trivial (it's a Poisson gain there).

## 5. PyTorch backend — same forward LL, gradient via autograd

The PyTorch backend computes the gradient w.r.t. the **raw** rate parameters (μ, λ, γ, t) via reverse-mode AD. For κ-parameterized Pólya nodes this matches the NumPy survival GAIN gradient exactly; for Poisson nodes (like the root in this example), the autograd ∂/∂r differs from the NumPy ∂/∂r̃ by the factor (1−ε) — that's the chain rule kicking in.

In [ ]:
import torch
from recount.torch_backend import corrected_log_likelihood_t, gradient_autograd

dt = torch.float64
gain_t   = torch.tensor(rates.gain,   dtype=dt)
loss_t   = torch.tensor(rates.loss,   dtype=dt)
dup_t    = torch.tensor(rates.dup,    dtype=dt)
length_t = torch.tensor(rates.length, dtype=dt)
profiles_t = torch.tensor(profiles, dtype=torch.long)

ll_t = corrected_log_likelihood_t(tree, gain_t, loss_t, dup_t, length_t, profiles_t, min_copies=1)
print(f"PyTorch corrected LL = {float(ll_t):.10f}")

grads = gradient_autograd(tree, gain_t, loss_t, dup_t, length_t, profiles_t, min_copies=1)
for k in ("gain", "loss", "dup", "length"):
    print(f"  ∂LL*/∂{k:6s} = {grads[k].numpy()}")

## 6. Maximum-likelihood estimation (optional)

PyTorch autograd makes it trivial to run gradient-based optimization. Below is a sketch using `torch.optim.LBFGS` to find the MLE of the rates.

In [ ]:
# Parameterize in log-space so rates stay positive. The duplication rate
# is bounded below the loss rate via dup_rate = loss_rate * sigmoid(z) so
# the model stays in the dup<loss regime where q<1 holds comfortably.
log_gain = gain_t.detach().clone().log().requires_grad_(True)
log_loss = loss_t.detach().clone().log().requires_grad_(True)
z_dup    = torch.zeros_like(dup_t).requires_grad_(True)

def neg_log_likelihood():
    g = log_gain.exp()
    l = log_loss.exp()
    d = l * torch.sigmoid(z_dup)
    return -corrected_log_likelihood_t(tree, g, l, d, length_t, profiles_t, min_copies=1)

optimizer = torch.optim.Adam([log_gain, log_loss, z_dup], lr=0.05)

for step in range(30):
    optimizer.zero_grad()
    loss = neg_log_likelihood()
    loss.backward()
    optimizer.step()
    if step % 5 == 0 or step == 29:
        print(f"step {step:>3d}: neg-LL = {float(loss):.6f}")

print()
print("optimized gain rates: ", log_gain.exp().detach().numpy().round(4))
print("optimized loss rates: ", log_loss.exp().detach().numpy().round(4))
print("optimized dup rates:  ", (log_loss.exp() * torch.sigmoid(z_dup)).detach().numpy().round(4))

## Reference

M. Csűrös. "Gain-loss-duplication models on a phylogeny: exact algorithms for computing the likelihood and its gradient." arXiv:2107.11440 (2021).